# Static EEG Data Wrangling

This notebook builds and analyzes a combined static dataset from CHB-MIT and Siena scalp EEG datasets

## Notebook goals

- Configure dataset and preprocessing settings.
- Bootstrap the repository so project modules can be imported.
- Resolve the dataset root containing `chbmit/` and `siena/`.
- Run the static dataset-building pipeline script.
- Load the generated dataset outputs for inspection and EDA.


## Running Instructions

This notebook works in both **VS Code** and **Google Colab**.

### Local (VS Code)
- Run all cells from top to bottom.

### Google Colab
- Upload one of the following files:
  - `master_eeg_dataset.parquet`
  - `master_eeg_dataset.csv`
- Then **skip to Step 3: Load the Dataset**

Use the file upload button in the left sidebar in Colab.

## Execution notes and dataset contract

This section explains how the notebook discovers data and what folder layout it expects.

Resolution order:

- `DATASET_ROOT_OVERRIDE` from the configuration cell below.
- `EEG_DATA_ROOT` from the environment.
- Local `./data` folders discovered upward from the current working directory.
- Google Drive locations when running in Colab.

The dataset root must contain the folders `chbmit/` and `siena/`.

Outputs from this notebook:

- `master_eeg_dataset.parquet` — main feature dataframe
- `master_eeg_dataset.csv` — CSV version of the dataset
- `master_eeg_dataset_X.npy` — feature matrix
- `master_eeg_dataset_y.npy` — labels (0 = interictal, 1 = preictal)
- `master_eeg_dataset_meta.csv` — metadata


## Configuration

This section keeps the settings most people are likely to tweak.

Inputs: an optional dataset root override, plus preprocessing and epoch parameters.

Output for the next step: notebook configuration variables used by the bootstrap and workflow cells.


In [33]:
# Optional explicit dataset root. Leave as None to use EEG_DATA_ROOT, local ./data,
# or Colab / Google Drive discovery.
DATASET_ROOT_OVERRIDE = "D:/"

# In Colab, allow the dataset resolver to mount Google Drive when the dataset is not local.
AUTO_MOUNT_GOOGLE_DRIVE = True

# Static dataset build configuration
EPOCH_DURATION_SEC = 10.0
EPOCH_OVERLAP_SEC = 0.0
TARGET_SFREQ = 256
PREICTAL_HORIZON_SEC = 600
POSTICTAL_EXCLUSION_SEC = 1800
INTERICTAL_GAP_SEC = 300

## Repo bootstrap

This step makes the `src/` package importable whether the notebook is running locally or in Google Colab.

Inputs: the current working directory and the optional `EEG_PROJECT_ROOT` environment variable.

Output for the next step: a resolved source path added to `sys.path`, plus helper functions for cleaner table displays.


In [34]:
from pathlib import Path
import os
import sys
import tempfile
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def is_repo_root(candidate: Path) -> bool:
    return (candidate / "pyproject.toml").exists() and (
        candidate / "src" / "eeg_project"
    ).exists()


def dedupe_paths(paths):
    deduped = []
    seen = set()
    for path in paths:
        key = str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped


def candidate_repo_roots():
    roots = []

    env_root = os.environ.get("EEG_PROJECT_ROOT")
    if env_root:
        roots.append(Path(env_root).expanduser())

    roots.extend([Path.cwd(), *Path.cwd().parents])

    if running_in_colab():
        roots.extend(
            [
                Path("/content"),
                Path("/content/eeg-Spr2026-CSCI7090"),
                Path("/content/drive/MyDrive"),
                Path("/content/drive/MyDrive/eeg-Spr2026-CSCI7090"),
                Path("/content/drive/MyDrive/Colab Notebooks"),
                Path("/content/drive/Shareddrives"),
            ]
        )

    roots.extend([Path.home(), Path.home() / "Developer", Path.home() / "Documents"])
    return dedupe_paths(path.expanduser() for path in roots)


def discover_repo_root() -> Path:
    checked = []

    for root in candidate_repo_roots():
        for candidate in [root, *root.parents]:
            checked.append(candidate)
            if is_repo_root(candidate):
                return candidate.resolve()

    search_roots = [root for root in candidate_repo_roots() if root.exists()]
    matches_checked = 0
    for search_root in search_roots:
        for pyproject_file in search_root.rglob("pyproject.toml"):
            matches_checked += 1
            candidate = pyproject_file.parent
            checked.append(candidate)
            if is_repo_root(candidate):
                return candidate.resolve()
            if matches_checked >= 300:
                break
        if matches_checked >= 300:
            break

    checked_display = "\n".join(f" - {path}" for path in dedupe_paths(checked)[:20])
    raise ModuleNotFoundError(
        "Could not locate the repository root.\n"
        "Local: open this notebook from the project workspace or set EEG_PROJECT_ROOT.\n"
        "Colab: clone the repo into /content or set EEG_PROJECT_ROOT to the mounted Drive repo path.\n\n"
        f"Checked:\n{checked_display}"
    )


def ensure_repo_src_on_path() -> Path:
    repo_root = discover_repo_root()
    src_dir = (repo_root / "src").resolve()
    resolved = str(src_dir)
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    return src_dir


def display_df(title: str, df: pd.DataFrame) -> None:
    display(Markdown(f"### {title}"))
    display(df)


def summarize_raw(raw, dataset_name: str, stage: str) -> dict:
    duration_sec = raw.n_times / raw.info["sfreq"] if raw.info["sfreq"] else np.nan
    return {
        "dataset": dataset_name,
        "stage": stage,
        "sampling_rate_hz": float(raw.info["sfreq"]),
        "n_channels": int(len(raw.ch_names)),
        "n_samples": int(raw.n_times),
        "duration_sec": round(float(duration_sec), 2),
        "first_channels": ", ".join(raw.ch_names[:6]),
    }


def summarize_epochs(
    dataset_name: str, epoch_array: np.ndarray, sampling_rate: float
) -> dict:
    return {
        "dataset": dataset_name,
        "epoch_count": int(epoch_array.shape[0]),
        "channel_count": int(epoch_array.shape[1]),
        "samples_per_epoch": int(epoch_array.shape[2]),
        "epoch_duration_sec": round(epoch_array.shape[2] / sampling_rate, 2),
        "sampling_rate_hz": float(sampling_rate),
    }


pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="mne")

src_dir = ensure_repo_src_on_path()
print("Using source path:", src_dir)
print("Running in Colab:", running_in_colab())

Using source path: C:\Developer\Python\EEG Project\eeg-Spr2026-CSCI7090\src
Running in Colab: False


## Step 1: Resolve the dataset root

This step finds the directory that directly contains both `chbmit/` and `siena/`.

Function used: `resolve_dataset_root`.

Inputs: `DATASET_ROOT_OVERRIDE` and `AUTO_MOUNT_GOOGLE_DRIVE`.

Output for the next step: `dataset_root` and a trace of every path that was checked.


In [35]:
from eeg_project.common.paths import resolve_dataset_root

if DATASET_ROOT_OVERRIDE:
    os.environ["EEG_DATA_ROOT"] = str(Path(DATASET_ROOT_OVERRIDE).expanduser())

dataset_root, checked_paths = resolve_dataset_root(
    DATASET_ROOT_OVERRIDE,
    mount_google_drive_if_needed=AUTO_MOUNT_GOOGLE_DRIVE,
)

print("Resolved dataset root:", dataset_root)
checked_paths_df = pd.DataFrame({"checked_path": [str(path) for path in checked_paths]})
display_df("Dataset Root Search Trace", checked_paths_df)

Resolved dataset root: D:\


### Dataset Root Search Trace

,checked_path
0,D:\


## Step 2: Build the Dataframe

This step puts both static datasets into a dataframe, by running the pipeline script named build_static_dataset

In [36]:
# Section 4 — Build merged dataset

import subprocess
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
output_root = repo_root / "artifacts" / "static"
output_root.mkdir(parents=True, exist_ok=True)

dataset_root_str = str(dataset_root)

cmd = [
    sys.executable,
    "-m",
    "scripts.build_static_dataset",
    "--dataset-root",
    dataset_root_str,
    "--output-root",
    str(output_root),
    "--epoch-duration",
    str(EPOCH_DURATION_SEC),
    "--epoch-overlap",
    str(EPOCH_OVERLAP_SEC),
    "--preictal-horizon-sec",
    str(PREICTAL_HORIZON_SEC),
    "--postictal-exclusion-sec",
    str(POSTICTAL_EXCLUSION_SEC),
    "--interictal-gap-sec",
    str(INTERICTAL_GAP_SEC),
]

print("Running command:")
print(" ".join(cmd))

# 🔥 THIS IS THE KEY FIX
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root / "src")

result = subprocess.run(
    cmd,
    cwd=repo_root,
    env=env,
    text=True,
    capture_output=True,
)

print("STDOUT:\n")
print(result.stdout)

if result.stderr:
    print("STDERR:\n")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Dataset build failed with exit code {result.returncode}")

print("\nBuild completed.")
print(f"Outputs saved in: {output_root}")
print("Files:")
for p in sorted(output_root.iterdir()):
    print(" -", p.name)

Running command:
c:\Users\arnol\AppData\Local\Programs\Python\Python313\python.exe -m scripts.build_static_dataset --dataset-root D:\ --output-root c:\Developer\Python\EEG Project\eeg-Spr2026-CSCI7090\artifacts\static --epoch-duration 10.0 --epoch-overlap 0.0 --preictal-horizon-sec 600 --postictal-exclusion-sec 1800 --interictal-gap-sec 300
STDOUT:

[DEBUG] About to load annotations
[DEBUG] Reading Siena annotation file: D:\siena\LICENSE.txt
[DEBUG] Reading Siena annotation file: D:\siena\SHA256SUMS.txt
[DEBUG] Siena annotation file had possible content but no parsed intervals: D:\siena\SHA256SUMS.txt
[DEBUG] Reading Siena annotation file: D:\siena\PN00\Seizures-list-PN00.txt
[DEBUG] Parsed 5 Siena seizure intervals from D:\siena\PN00\Seizures-list-PN00.txt
[DEBUG] Reading Siena annotation file: D:\siena\PN01\Seizures-list-PN01.txt
[DEBUG] Siena annotation file had possible content but no parsed intervals: D:\siena\PN01\Seizures-list-PN01.txt
[DEBUG] Reading Siena annotation file: D:\s

## Step 3: Load the Dataset

In [37]:
from pathlib import Path
import pandas as pd
import sys

possible_parquet_paths = [
    Path("artifacts/static/master_eeg_dataset.parquet"),
    Path("master_eeg_dataset.parquet"),
]

possible_csv_paths = [
    Path("artifacts/static/master_eeg_dataset.csv"),
    Path("master_eeg_dataset.csv"),
]

parquet_path = next((p for p in possible_parquet_paths if p.exists()), None)
csv_path = next((p for p in possible_csv_paths if p.exists()), None)

if parquet_path is not None:
    print("Loading dataset from:", parquet_path)
    df_all = pd.read_parquet(parquet_path)

elif csv_path is not None:
    print("Loading dataset from:", csv_path)
    try:
        df_all = pd.read_csv(csv_path)
    except Exception:
        df_all = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")

else:
    raise FileNotFoundError(
        "Could not find master_eeg_dataset.parquet or master_eeg_dataset.csv.\n"
        "Place the dataset in 'artifacts/static/' or upload it directly to the working directory."
    )

print("Shape:", df_all.shape)
display(df_all.groupby('dataset').head(5))

Loading dataset from: artifacts\static\master_eeg_dataset.parquet
Shape: (354478, 29)


,source_type,dataset,subject_id,session_id,record_id,edf_path,source_file_name,epoch_index,window_start_sec,window_end_sec,...,std,min,max,range,energy,rms,abs_mean,n_channels,n_samples,sfreq
0,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000000,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,0,0.0,10.0,...,0.000046,-0.000339,0.000476,0.000815,0.000119,0.000046,0.000028,22,2560,256.0
1,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000001,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,1,10.0,20.0,...,0.000040,-0.000467,0.000461,0.000928,0.000089,0.000040,0.000024,22,2560,256.0
2,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000002,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,2,20.0,30.0,...,0.000039,-0.000414,0.000454,0.000868,0.000085,0.000039,0.000024,22,2560,256.0
3,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000003,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,3,30.0,40.0,...,0.000028,-0.000182,0.000444,0.000626,0.000045,0.000028,0.000018,22,2560,256.0
4,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000004,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,4,40.0,50.0,...,0.000034,-0.000361,0.000320,0.000682,0.000067,0.000034,0.000021,22,2560,256.0
329893,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000000,D:\siena\PN00\PN00-1.edf,PN00-1.edf,0,0.0,10.0,...,0.000079,-0.000746,0.002810,0.003556,0.000554,0.000079,0.000024,35,2560,256.0
329894,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000182,D:\siena\PN00\PN00-1.edf,PN00-1.edf,182,1820.0,1830.0,...,0.000070,-0.000844,0.002438,0.003282,0.000443,0.000070,0.000018,35,2560,256.0
329895,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000183,D:\siena\PN00\PN00-1.edf,PN00-1.edf,183,1830.0,1840.0,...,0.000068,-0.000810,0.002415,0.003226,0.000410,0.000068,0.000017,35,2560,256.0
329896,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000184,D:\siena\PN00\PN00-1.edf,PN00-1.edf,184,1840.0,1850.0,...,0.000070,-0.000815,0.002430,0.003245,0.000437,0.000070,0.000018,35,2560,256.0
329897,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000185,D:\siena\PN00\PN00-1.edf,PN00-1.edf,185,1850.0,1860.0,...,0.000069,-0.000814,0.002401,0.003215,0.000426,0.000069,0.000017,35,2560,256.0
